# nb12.2 — SSRN Metadata Generator: JEL Codes, Keywords, and Abstract-Diff Control

*Week 12, Chapter 12.2.* Posting a working paper to SSRN looks like a five-minute
form. It is not. The five fields you fill in — title, abstract, JEL codes, keywords,
and "what version is this" — determine who finds your paper, how the citation
trackers categorize it, and whether your future self can reconstruct which draft
the world is currently looking at.

This notebook generates all five fields from a single parameter dictionary, so
the same paper produces consistent metadata across SSRN, the Schar repository,
ORCID, and your CV. We also build a small abstract-version-control system that
diffs draft N against draft N-1 so you can answer the inevitable advisor
question: *"What changed since I last read this?"*


In [ ]:
import matplotlib
matplotlib.use("Agg")

import difflib
import hashlib
import json
import re
from dataclasses import asdict, dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd

rng = np.random.default_rng(42)


## 1. The parameter dictionary

We start from a single source of truth: a dataclass holding everything SSRN
needs. Devon's crypto-payments paper is the worked example. Every other
artifact in this notebook is *derived* from this object, so if Devon decides
to retitle the paper, he changes one line and everything downstream updates.


In [ ]:
@dataclass
class PaperMeta:
    title: str
    authors: list
    affiliation: str
    abstract: str
    primary_field: str        # e.g. "Financial Economics"
    secondary_fields: list = field(default_factory=list)
    methods: list = field(default_factory=list)        # e.g. "diff-in-diff"
    data_sources: list = field(default_factory=list)
    asset_classes: list = field(default_factory=list)
    version: str = "v1"
    draft_date: str = "2026-09-11"

devon = PaperMeta(
    title="Stablecoin Adoption and Cross-Border Remittance Fees: Evidence from Chainalysis 2020-2024",
    authors=["Devon Park", "Lei Gao"],
    affiliation="George Mason University",
    abstract=(
        "We use a panel of 28 remittance corridors from 2020 to 2024 to estimate the effect "
        "of stablecoin adoption on cross-border remittance fees. Using a difference-in-differences "
        "design that exploits the staggered launch of USDC corridor support, we find that a one "
        "standard deviation increase in stablecoin transaction volume reduces average remittance "
        "fees by 0.42 percentage points, concentrated in corridors with weak banking competition. "
        "Effects are robust to controlling for mobile money penetration and remittance flow size, "
        "and are absent in corridors where stablecoin support was announced but not launched."
    ),
    primary_field="Financial Economics",
    secondary_fields=["International Finance", "Banking"],
    methods=["difference-in-differences", "staggered adoption", "two-way fixed effects"],
    data_sources=["Chainalysis", "World Bank Remittance Prices Worldwide", "BIS"],
    asset_classes=["stablecoins", "cryptocurrency", "remittances"],
)
asdict(devon)


## 2. JEL codes: the field's Dewey Decimal

The Journal of Economic Literature classification is a four-character code
(letter + three digits) used by every economics database to route papers. SSRN
asks for between one and six codes; conventional practice is three to five with
the *primary* code listed first.

We do not pretend our code lookup is the full JEL tree — that lives at
[aeaweb.org/econlit/jelCodes.php](https://www.aeaweb.org/econlit/jelCodes.php).
What we ship is a curated subset of about twenty codes that cover the topics
our students actually write about, plus a string-matching rule that turns a
paper's `methods`, `asset_classes`, and `secondary_fields` into a ranked list
of candidate codes.


In [ ]:
JEL_CATALOG = pd.DataFrame([
    ("G14", "Information and Market Efficiency; Event Studies; Insider Trading",
     ["event study", "market efficiency", "insider"]),
    ("G15", "International Financial Markets",
     ["international", "cross-border", "remittance", "fx"]),
    ("G18", "Government Policy and Regulation (Financial Markets)",
     ["regulation", "policy", "cfpb", "sec"]),
    ("G21", "Banks; Depository Institutions; Mortgages",
     ["bank", "mortgage", "hmda", "deposit", "lending"]),
    ("G23", "Non-bank Financial Institutions; Mutual Funds",
     ["mutual fund", "etf", "non-bank", "fintech"]),
    ("G24", "Investment Banking; Venture Capital; Brokerage; Ratings",
     ["venture", "ipo", "underwriting", "rating"]),
    ("G28", "Government Policy and Regulation (Financial Institutions)",
     ["regulation", "supervision", "stress test"]),
    ("G29", "Financial Institutions and Services - Other",
     ["stablecoin", "cryptocurrency", "crypto", "blockchain"]),
    ("G32", "Financing Policy; Capital and Ownership Structure",
     ["leverage", "capital structure", "ownership"]),
    ("G34", "Mergers; Acquisitions; Restructuring",
     ["merger", "acquisition", "spinoff"]),
    ("G41", "Behavioral Finance: Role of Psychology in Decision-Making",
     ["behavioral", "sentiment", "overconfidence"]),
    ("C21", "Single Equation Models; Cross-Sectional",
     ["cross-section", "regression"]),
    ("C23", "Panel Data Models",
     ["panel", "fixed effects", "two-way", "difference-in-differences", "staggered"]),
    ("C26", "Instrumental Variables",
     ["instrument", "iv", "natural experiment"]),
    ("C45", "Neural Networks and Related Topics",
     ["neural", "machine learning", "deep learning"]),
    ("D14", "Household Saving; Personal Finance",
     ["household", "personal finance", "credit card", "bnpl"]),
    ("J15", "Economics of Minorities, Races, and Immigrants",
     ["fair lending", "discrimination", "race", "minority"]),
    ("Q54", "Climate; Natural Disasters; Global Warming",
     ["climate", "carbon", "emissions", "flood", "hurricane"]),
    ("O34", "Intellectual Property and Intellectual Capital",
     ["patent", "trademark", "intellectual property"]),
], columns=["code", "description", "triggers"])

JEL_CATALOG.head()


## 3. The ranker

For each candidate code we score how strongly the paper triggers it. The score
is the count of trigger phrases found in the lowercased concatenation of the
methods, asset classes, secondary fields, and the abstract itself. Ties broken
alphabetically by code, so the ranking is deterministic.


In [ ]:
def rank_jel_codes(meta, catalog, top_k=5):
    """Score each catalog row against the paper text; return top_k by score."""
    haystack = " ".join([
        meta.abstract,
        " ".join(meta.methods),
        " ".join(meta.asset_classes),
        " ".join(meta.secondary_fields),
        meta.primary_field,
    ]).lower()
    scores, hits = [], []
    for triggers in catalog["triggers"]:
        matched = [t for t in triggers if t in haystack]
        scores.append(len(matched))
        hits.append(", ".join(matched) if matched else "")
    out = catalog.assign(score=scores, matched=hits)
    out = out.sort_values(["score", "code"], ascending=[False, True])
    return out.head(top_k).reset_index(drop=True)

jel_ranked = rank_jel_codes(devon, JEL_CATALOG, top_k=5)
jel_ranked[["code", "description", "score", "matched"]]


**Why this works.** Devon's abstract uses the words *"difference-in-differences"*,
*"staggered"*, *"cross-border"*, *"remittance"*, and *"stablecoin"*. The ranker
finds these and scores `C23`, `G15`, and `G29` highly — exactly what an editor
specializing in international banking would expect to see.

**When it fails.** If your abstract is too abstract (no asset names, no method
verbs, lots of *"we investigate the relationship between"*), the ranker scores
everything zero and falls back to alphabetical. Test fix: write a richer abstract.


## 4. Keywords

SSRN allows up to six keywords. Best practice is to repeat a few terms from the
title (because search), introduce a few synonyms (because not everyone calls a
stablecoin a stablecoin), and avoid the obvious filler words the abstract
already contains. We mine candidate keywords from the abstract and asset classes,
deduplicate, lowercase, and cap at six.


In [ ]:
STOPWORDS = set("""
the a an and or but for of in on at to from with by we our this that these those
is are was were be been being effect effects evidence using uses paper study
result results find finds analysis approach show shows
""".split())

def extract_keywords(meta, k=6):
    """Return up to k lowercase keywords drawn from the abstract and asset classes."""
    # Tokenize abstract into lowercase words, strip punctuation
    tokens = re.findall(r"[a-z][a-z\-]+", meta.abstract.lower())
    # Bigrams catch terms like "stablecoin adoption"
    bigrams = [f"{tokens[i]} {tokens[i+1]}" for i in range(len(tokens) - 1)]
    counts = pd.Series(tokens + bigrams).value_counts()
    # Drop stopwords and very short tokens
    counts = counts[~counts.index.isin(STOPWORDS)]
    counts = counts[counts.index.str.len() >= 4]
    # Seed the list with the asset classes (they almost always belong)
    keywords = list(dict.fromkeys(meta.asset_classes))
    for w in counts.index:
        if len(keywords) >= k:
            break
        if w in keywords:
            continue
        keywords.append(w)
    return keywords[:k]

keywords = extract_keywords(devon, k=6)
keywords


## 5. The SSRN form, fully filled

Now we stitch JEL codes and keywords onto the metadata and emit the SSRN
submission form as a dictionary. In practice you would paste these fields into
the SSRN website (no public API exists for student deposits). The point is that
*the dictionary is the canonical version*. The SSRN form is a downstream
artifact; you regenerate it any time you retitle the paper or refresh the JEL
catalog.


In [ ]:
def ssrn_form(meta, jel_df, keywords):
    return {
        "title": meta.title,
        "authors": meta.authors,
        "affiliation": meta.affiliation,
        "abstract": meta.abstract,
        "jel_codes": jel_df["code"].tolist(),
        "keywords": keywords,
        "primary_field": meta.primary_field,
        "version": meta.version,
        "draft_date": meta.draft_date,
        "abstract_word_count": len(re.findall(r"\b\w+\b", meta.abstract)),
        "abstract_sha256": hashlib.sha256(meta.abstract.encode("utf-8")).hexdigest(),
    }

form = ssrn_form(devon, jel_ranked, keywords)
print(json.dumps(form, indent=2))


## 6. Abstract version control

Now the second half of this notebook. SSRN tracks "Version X" as a paragraph
of metadata, but it does **not** show you a diff. Your advisor reads v2,
suggests changes, you upload v3, and three weeks later neither of you remembers
what changed.

We fix this with a small library of three functions:

1. `save_version(text, label)` — hash the abstract, store it, and stamp it
   with a label like `"v1-feb-15"`.
2. `diff_versions(a_label, b_label)` — print a unified diff between two stored
   versions.
3. Display the version table — show every stored version with its date, label,
   hash, and length so you can pick which two to diff.

The storage is a JSON file in the workspace; nothing leaves your laptop.


In [ ]:
VERSIONS_DIR = Path("/tmp/nb12_2_versions")
VERSIONS_DIR.mkdir(exist_ok=True)
VERSIONS_FILE = VERSIONS_DIR / "abstract-versions.json"

def _load_store():
    if not VERSIONS_FILE.exists():
        return []
    return json.loads(VERSIONS_FILE.read_text())

def _write_store(store):
    VERSIONS_FILE.write_text(json.dumps(store, indent=2))

def save_version(text, label, date):
    store = _load_store()
    entry = {
        "label": label,
        "date": date,
        "sha256": hashlib.sha256(text.encode("utf-8")).hexdigest(),
        "word_count": len(re.findall(r"\b\w+\b", text)),
        "text": text,
    }
    # Idempotent: if the same label already exists, replace it.
    store = [e for e in store if e["label"] != label]
    store.append(entry)
    _write_store(store)
    return entry

# Reset the store so this notebook is hermetic
_write_store([])


### Three versions of Devon's abstract

We hard-code three drafts so the diff has something to chew on. Notice how v2
adds a quantitative result and v3 sharpens the causal language from *"is
associated with"* to *"reduces"*.


In [ ]:
v1 = (
    "We examine whether the rise of stablecoins is associated with changes in "
    "remittance fees in a panel of 28 corridors. Initial results suggest that "
    "adoption may be linked to lower fees, though causal interpretation is limited."
)
v2 = (
    "We examine whether the rise of stablecoins is associated with changes in "
    "remittance fees in a panel of 28 corridors from 2020 to 2024. Using a "
    "difference-in-differences design, a one standard deviation increase in "
    "stablecoin transaction volume is associated with remittance fees that are "
    "0.4 percentage points lower on average."
)
v3 = devon.abstract  # the polished version-3 abstract from cell 1

save_version(v1, "v1-feb-15", "2026-02-15")
save_version(v2, "v2-mar-22", "2026-03-22")
save_version(v3, "v3-sep-11", "2026-09-11")

store = _load_store()
pd.DataFrame([{k: v for k, v in e.items() if k != "text"} for e in store])


### Unified diff between v2 and v3

`difflib.unified_diff` is the same algorithm `git diff` uses. Lines starting
with `-` were removed, lines with `+` were added, lines starting with a space
are unchanged context.


In [ ]:
def diff_versions(label_a, label_b):
    store = _load_store()
    by_label = {e["label"]: e for e in store}
    if label_a not in by_label or label_b not in by_label:
        raise KeyError(f"Unknown label(s); available: {list(by_label)}")
    a = by_label[label_a]["text"].splitlines(keepends=True)
    b = by_label[label_b]["text"].splitlines(keepends=True)
    lines = list(difflib.unified_diff(a, b, fromfile=label_a, tofile=label_b, lineterm=""))
    return "\n".join(lines) if lines else "(identical)"

print(diff_versions("v2-mar-22", "v3-sep-11"))


For prose, line-based diffs are coarse. A more useful view for an abstract is
**word-level** — the smallest unit of change. We build a small word-diff with
`difflib.ndiff` that returns each token tagged with whether it was kept (`  `),
added (`+ `), or removed (`- `).


In [ ]:
def word_diff(label_a, label_b):
    store = _load_store()
    by_label = {e["label"]: e for e in store}
    a_words = re.findall(r"\S+|\s+", by_label[label_a]["text"])
    b_words = re.findall(r"\S+|\s+", by_label[label_b]["text"])
    rows = []
    for token in difflib.ndiff(a_words, b_words):
        tag, word = token[:2], token[2:]
        if word.strip() == "":
            continue
        if tag in ("  ", "- ", "+ "):
            rows.append({"tag": tag.strip() or "=", "token": word})
    return pd.DataFrame(rows)

wd = word_diff("v2-mar-22", "v3-sep-11")
# Show added and removed tokens only
wd[wd["tag"].isin(["-", "+"])].head(20)


## 7. Putting it together: the SSRN submission summary

The final cell is what you screenshot and paste into your advisor email the
morning of the deadline. It bundles the SSRN-form dictionary, the three-line
version history, and the headline diff statistics.


In [ ]:
def submission_summary(meta):
    jel = rank_jel_codes(meta, JEL_CATALOG, top_k=5)
    kws = extract_keywords(meta, k=6)
    form = ssrn_form(meta, jel, kws)
    store = _load_store()
    lines = [
        f"TITLE: {form['title']}",
        f"AUTHORS: {', '.join(form['authors'])}",
        f"AFFILIATION: {form['affiliation']}",
        f"JEL CODES (primary first): {', '.join(form['jel_codes'])}",
        f"KEYWORDS: {', '.join(form['keywords'])}",
        f"ABSTRACT ({form['abstract_word_count']} words; "
        f"sha256={form['abstract_sha256'][:12]}...)",
        f"VERSION HISTORY ({len(store)} versions):",
    ]
    for e in store:
        lines.append(f"  {e['label']:<14} {e['date']}  {e['word_count']:>3} words  {e['sha256'][:12]}...")
    return "\n".join(lines)

print(submission_summary(devon))


## 8. When it fails

**JEL drift.** The JEL list above is a curated subset. Real SSRN forms accept
hundreds of codes. If your paper is in a niche subfield (say, behavioral
corporate finance with text-mining), the top-5 ranker will hit zero or one
match. Solution: extend `JEL_CATALOG` with the codes you actually want and
re-rank.

**Keyword bigrams that are noise.** The bigram extractor produces phrases like
*"transaction volume"* and *"average remittance"* which are fine, but also
sometimes *"are absent"* — clearly junk. The next-version fix is a part-of-speech
filter using `spacy`, but that requires a 500 MB model. For undergraduate work,
a manual eyeball of the keyword list is enough.

**Abstract diffs across copy-edit waves.** If you ran your abstract through a
heavy copy edit between v2 and v3, almost every line changes and the diff is
unreadable. The fix is to commit a copy-edit pass as its own version (`v2.1`)
so substantive revisions show up cleanly afterward.


## 9. Your turn

1. **Add Maya's paper.** Build a `PaperMeta` for Maya's HMDA fair-lending
   paper. Confirm the ranker picks `J15` and `G21` in the top three.

2. **Tighten the stopword list.** Add ten more domain stopwords (try *"using"*,
   *"effect"*, *"results"*) and confirm the keyword list now leads with
   substantive terms.

3. **Diff-against-latest.** Write `diff_against_latest(text)` that takes a new
   draft string, saves it as the next version automatically (label `vN`), and
   prints the diff against the previous version.

4. **Render to a Markdown email.** Turn `submission_summary` into a function
   that emits a Markdown-formatted email body suitable for pasting into Gmail.
